# STAIR-v1 — Stepwise Graph Contrastive Learning (STAIR-GCL)

**Mô hình:** STAIR-v1 (STAIR + Graph Contrastive Learning)
**Ý tưởng 1 (Core & Loss):**  
1. Loại bỏ cạnh ngẫu nhiên (Edge Dropout $p_{drop}=0.2$) tạo 2 view đồ thị phụ trợ $\mathcal{G}'$ và $\mathcal{G}''$.  
2. Áp dụng InfoNCE Contrastive Loss **chỉ cho 32 chiều đầu** (Collaborative Subspace), giúp lọc nhiễu tương tác.  
3. Giữ nguyên 32 chiều sau (Multimodal Subspace) bảo tồn thuộc tính hình ảnh/văn bản qua FSC không bị oversmoothing.

**Datasets thử nghiệm:** `baby` (Amazon2014Baby_550_MMRec) & `sports` (Amazon2014Sports_550_MMRec)

## Cell 1 — Thiết lập Môi trường & Kéo Mã nguồn Public từ GitHub

In [ ]:
import os, shutil

os.chdir('/kaggle/working')

github_user = 'ThanhChuong12'
repo_name   = 'STAIR-Enhanced'

if os.path.exists(repo_name):
    shutil.rmtree(repo_name)
    print('🗑️ Da xoa thu muc ' + repo_name + ' cu.')

git_url = f'https://github.com/{github_user}/{repo_name}.git'
print('⬇️ Dang clone public repo: ' + git_url)
os.system('git clone ' + git_url)

print('⚙️ Dang cai dat thu vien (freerec, torch_geometric, nvidia-ml-py)...')
os.system('pip install nvidia-ml-py -q')
os.system('pip install torchdata==0.6.1 --no-deps -q')
os.system('pip install freerec==0.8.3 -q')
os.system('pip install torch_geometric -q')

os.chdir(f'/kaggle/working/{repo_name}')
print('Thu muc hien tai: ' + os.getcwd())
for f in ['main.py', 'main_v1.py']:
    exists = os.path.exists(f)
    print('  [' + ('OK' if exists else 'MISSING!') + '] ' + f)
print('✅ Moi truong va repo da san sang!')

## Cell 2 — Chuẩn bị Dữ liệu Linh hoạt (`baby` & `sports`)

In [ ]:
import os, shutil

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs('data/baby', exist_ok=True)
os.makedirs('data/sports', exist_ok=True)

def smart_copy_dataset(kw, full_name, local_dest):
    kaggle_dest = os.path.join(DATA_ROOT, full_name)
    found = 0
    for root, _, files in os.walk('/kaggle/input'):
        if kw.lower() in root.lower():
            for f in files:
                if f.endswith(('.npy', '.pkl', '.txt')):
                    src_f = os.path.join(root, f)
                    shutil.copy(src_f, os.path.join(local_dest, f))
                    os.makedirs(kaggle_dest, exist_ok=True)
                    shutil.copy(src_f, os.path.join(kaggle_dest, f))
                    found += 1
    if found > 0:
        print(f"  [+] Da copy thanh cong {found} files cho [{full_name}]")
    else:
        print(f"  [!] CANH BAO: Khong tim thay files chua tu khoa '{kw}' trong /kaggle/input!")

print('📁 Dang tim va copy du lieu an toan...')
smart_copy_dataset('baby',   'Amazon2014Baby_550_MMRec',   'data/baby')
smart_copy_dataset('sports', 'Amazon2014Sports_550_MMRec', 'data/sports')
print('\n✅ Du lieu da san sang!')

## Cell 3 — Kiểm tra Môi trường & Mã nguồn

In [ ]:
import torch, platform, os

print('='*60)
print('THONG TIN MOI TRUONG')
print('='*60)
print('  Python     : ' + platform.python_version())
print('  PyTorch    : ' + torch.__version__)
print('  CUDA avail : ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    print('  GPU name   : ' + torch.cuda.get_device_name(0))
print('  Working dir: ' + os.getcwd())
print('='*60)

## Cell 4 — Huấn luyện STAIR Baseline & STAIR-v1 (STAIR-GCL) trên `baby` & `sports`

> **Đã fix:** Bổ sung `--config` để FreeRec tính toán đúng chỉ số Recall@10, Recall@20, NDCG@10, NDCG@20.

In [ ]:
import time, subprocess, os

runs = [
    {
        'key': 'baby_baseline',
        'script': 'main.py',
        'dataset': 'Amazon2014Baby_550_MMRec',
        'weight_decay': '0.3',
        'gamma': '0.1',
        'extra_args': ['--config', 'configs/Amazon2014Baby_550_MMRec.yaml']
    },
    {
        'key': 'baby_v1',
        'script': 'main_v1.py',
        'dataset': 'Amazon2014Baby_550_MMRec',
        'weight_decay': '0.3',
        'gamma': '0.1',
        'extra_args': ['--config', 'configs/Amazon2014Baby_550_MMRec.yaml', '--cl-weight', '1e-3', '--edge-drop', '0.2', '--cl-temp', '0.2']
    },
    {
        'key': 'sports_baseline',
        'script': 'main.py',
        'dataset': 'Amazon2014Sports_550_MMRec',
        'weight_decay': '0.1',
        'gamma': '0.2',
        'extra_args': ['--config', 'configs/Amazon2014Sports_550_MMRec.yaml']
    },
    {
        'key': 'sports_v1',
        'script': 'main_v1.py',
        'dataset': 'Amazon2014Sports_550_MMRec',
        'weight_decay': '0.1',
        'gamma': '0.2',
        'extra_args': ['--config', 'configs/Amazon2014Sports_550_MMRec.yaml', '--cl-weight', '1e-3', '--edge-drop', '0.2', '--cl-temp', '0.2']
    },
]

results_log = {}

for run in runs:
    key = run['key']
    script = run['script']
    dataset = run['dataset']
    print('\n' + '='*60)
    print(f'🚀 DANG HUAN LUYEN: {key.upper()} ({script})')
    print('='*60)

    cmd = [
        'python', script,
        '--dataset',      dataset,
        '--root',         '/kaggle/data',
        '--weight-decay', run['weight_decay'],
        '--gamma',        run['gamma'],
        '--epochs',       '500',
        '--eval-freq',    '5',
    ] + run['extra_args']

    print('Lenh: ' + ' '.join(cmd))
    start_time = time.time()
    res = subprocess.run(cmd, capture_output=True, text=True, cwd='/kaggle/working/STAIR-Enhanced')
    elapsed = time.time() - start_time

    if res.returncode != 0:
        print(f'❌ LOI {key.upper()}:')
        print(res.stderr[-3000:])
        results_log[key] = {'status': 'FAIL', 'log': res.stderr[-2000:], 'time': elapsed}
    else:
        print(f'✅ HOAN THANH {key.upper()} trong {round(elapsed, 1)}s!')
        results_log[key] = {'status': 'OK', 'log': res.stdout, 'time': elapsed}

print('\n' + '='*60)
print('TAT CA PHIEN HUAN LUYEN HOAN THANH!')
print('='*60)

## Cell 5 — Đọc Metrics Trực tiếp từ `logs/.../log.txt` & Vẽ Biểu Đồ So Sánh

In [ ]:
import re, os, glob
import matplotlib.pyplot as plt
import numpy as np

def parse_log_txt_file(fpath):
    metrics = {}
    try:
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as fp:
            text = fp.read()
            if not text.strip(): return metrics
            
            lines = text.splitlines()
            for i, line in enumerate(lines):
                if 'Recall@10' in line or 'Recall@20' in line or 'NDCG@20' in line:
                    headers = [p.strip() for p in line.split('|') if p.strip()]
                    for j in range(i+1, min(i+10, len(lines))):
                        row = lines[j]
                        if '|' in row and not '---' in row and not 'Epoch' in row:
                            vals = [p.strip() for p in row.split('|') if p.strip()]
                            for h, v in zip(headers, vals):
                                if h in ['Recall@1', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
                                    try:
                                        val = float(v)
                                        if h not in metrics or val > metrics[h]:
                                            metrics[h] = val
                                    except: pass
            
            for m_name in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
                pattern = re.compile(rf'{m_name}\s*[:=\s]\s*([0-9.]+)')
                for match in pattern.finditer(text):
                    try:
                        val = float(match.group(1))
                        if m_name not in metrics or val > metrics[m_name]:
                            metrics[m_name] = val
                    except: pass
    except Exception as e: pass
    return metrics

def scan_exact_freerec_logs(base_dir='/kaggle/working/STAIR-Enhanced'):
    summary = {}
    log_files = glob.glob(os.path.join(base_dir, 'logs', '**', 'log.txt'), recursive=True)
    print(f'🔍 Dang quet {len(log_files)} file log.txt trong thu muc logs/...')
    
    for fpath in log_files:
        norm_path = fpath.replace('\\', '/').lower()
        is_v1 = ('stair-v1' in norm_path) or ('v1' in norm_path)
        is_baby = ('baby' in norm_path)
        is_sports = ('sports' in norm_path)
        
        run_key = None
        if is_baby:
            run_key = 'baby_v1' if is_v1 else 'baby_baseline'
        elif is_sports:
            run_key = 'sports_v1' if is_v1 else 'sports_baseline'
            
        if run_key:
            m = parse_log_txt_file(fpath)
            if m:
                summary[run_key] = m
                print(f"  [+] Tim thay tu: {os.path.basename(os.path.dirname(fpath))}/log.txt -> {run_key}: {m}")
    return summary

extracted = scan_exact_freerec_logs()

print('\n' + '='*70)
print('TONG HOP KET QUA STAIR BASELINE VS STAIR-v1')
print('='*70)
for k in ['baby_baseline', 'baby_v1', 'sports_baseline', 'sports_v1']:
    res = extracted.get(k, {})
    print(f"{k.upper().ljust(18)}: {res}")
print('='*70)

# Ve Bieu Do So Sanh
datasets = ['baby', 'sports']
metric_keys = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('STAIR Baseline vs STAIR-v1 (STAIR-GCL) Performance Comparison', fontsize=15, fontweight='bold')

for idx, ds in enumerate(datasets):
    ax = axes[idx]
    base_key = f'{ds}_baseline'
    v1_key   = f'{ds}_v1'
    
    base_vals = [extracted.get(base_key, {}).get(m, 0.0) for m in metric_keys]
    v1_vals   = [extracted.get(v1_key, {}).get(m, 0.0) for m in metric_keys]
    
    x = np.arange(len(metric_keys))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, base_vals, width, label='STAIR Baseline', color='#3498DB', alpha=0.85)
    bars2 = ax.bar(x + width/2, v1_vals, width, label='STAIR-v1 (GCL)', color='#2ECC71', alpha=0.85)
    
    for bar in bars1 + bars2:
        h = bar.get_height()
        if h > 0:
            ax.annotate(f'{h:.4f}', xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
    
    ax.set_title(f'Dataset: {ds.upper()}', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_keys, fontsize=10)
    ax.set_ylabel('Metric Value')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved chart to: /kaggle/working/stair_v1_comparison.png')